# 07 - Minimal Representation Theory

## The Central Question

Given the energy integral:

$$E = \int \int \varepsilon(r_i, r_j) \cdot P(\text{contact} | r_i, r_j) \, dr_i \, dr_j$$

**What is the MINIMAL representation $\theta$ such that $E \approx f(\theta)$?**

This is the "sufficient statistic" for phase separation thermodynamics.

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import matplotlib.pyplot as plt
from src.sequences import FUS_LCD_SEQUENCE, VARIANTS
from src.minimal_representation import (
    compute_total_energy_integral,
    compute_energy_with_contact_probability,
    generate_shuffled_sequences,
)

## Key Insight: Two Weighting Schemes

### 1. Uniform Weighting (Mean-Field)
$$E_{\text{uniform}} = \sum_{i < j} \varepsilon(a_i, a_j)$$

**Result**: Depends ONLY on composition → 20D representation is EXACT

### 2. Contact Probability Weighting (Polymer Physics)
$$E_{\text{contact}} = \sum_{i < j} \varepsilon(a_i, a_j) \cdot P(|i-j|)$$

where $P(d) \propto d^{-3/2}$ for Gaussian chain.

**Result**: Depends on sequence ARRANGEMENT → need more information

In [ ]:
# Demonstrate the difference
np.random.seed(42)
shuffled = generate_shuffled_sequences(FUS_LCD_SEQUENCE, 100)

E_uniform = [compute_total_energy_integral(s) for s in shuffled]
E_contact = [compute_energy_with_contact_probability(s) for s in shuffled]

print("UNIFORM weighting (all pairs):")
print(f"  Std across shuffles: {np.std(E_uniform):.6f}")
print(f"  → Composition is SUFFICIENT")
print()
print("CONTACT weighting (polymer physics):")
print(f"  Std across shuffles: {np.std(E_contact):.4f}")
print(f"  → Arrangement MATTERS")

## The Minimal Representation

For contact-weighted energy, the minimal representation is:

$$\theta = \{ n_{\alpha\beta}(d) : \alpha, \beta \in \text{stickers}, \, d \leq d_{\max} \}$$

where $n_{\alpha\beta}(d)$ = number of $\alpha$-$\beta$ pairs at separation $d$.

### Simplified Version (Sticker-Spacer Model)

Collapse to just "sticker" vs "non-sticker":

$$\theta = \{ n_{SS}(d) : d \leq d_{\max} \}$$

This is the **sticker-sticker pair distribution** by sequence separation.

In [ ]:
def sticker_pair_distribution(sequence, d_max=30):
    """Count sticker-sticker pairs at each separation."""
    stickers = set('YFWRK')
    n = len(sequence)
    
    counts = np.zeros(d_max)
    for i in range(n):
        if sequence[i] not in stickers:
            continue
        for j in range(i + 1, min(n, i + d_max)):
            d = j - i
            if sequence[j] in stickers:
                counts[d] += 1
    return counts

# Compare original vs shuffled
dist_orig = sticker_pair_distribution(shuffled[0])
dist_shuffled = np.mean([sticker_pair_distribution(s) for s in shuffled[1:]], axis=0)

fig, ax = plt.subplots(figsize=(10, 5))
d = np.arange(len(dist_orig))
ax.bar(d - 0.2, dist_orig, width=0.4, label='Original FUS', color='#E74C3C')
ax.bar(d + 0.2, dist_shuffled, width=0.4, label='Mean of shuffles', color='#3498DB')
ax.set_xlabel('Sequence separation d')
ax.set_ylabel('Number of sticker-sticker pairs')
ax.set_title('Sticker-Sticker Pair Distribution: θ(sequence)')
ax.legend()
plt.tight_layout()
plt.show()

## The Energy Integral Decomposition

The key equation:

$$E_{\text{contact}} = \sum_{d=1}^{d_{\max}} g(d) \cdot P(d)$$

where:
- $g(d) = \sum_{|i-j|=d} \varepsilon(a_i, a_j)$ = total interaction energy at separation $d$
- $P(d) \propto d^{-3/2}$ = contact probability

The representation $\theta = \{g(d)\}$ is the **energy-separation histogram**.

## Summary: The Hierarchy of Representations

| Representation | Dimension | Sufficient for |
|---------------|-----------|----------------|
| Full sequence | N | Everything |
| Pair counts by type | 210 | $E_{\text{uniform}}$ |
| Composition | 20 | $E_{\text{uniform}}$ |
| Energy-separation histogram | ~30 | $E_{\text{contact}}$ |
| Sticker pair distribution | ~30 | $E_{\text{contact}}$ (approx) |
| Sticker count + spacing stats | 5 | Partial (~50% R²) |
| Sticker count alone | 1 | Not sufficient |

### The Answer

**For phase separation physics**, the minimal sufficient representation is:

$$\boxed{\theta = \{n_{SS}(d) : d = 1, \ldots, d_{\max}\}}$$

with $d_{\max} \approx 30$ (the contact range).

This is **30-dimensional** - much smaller than the full sequence (N=214) or all pair counts (210), but larger than simple sticker counting (1D).